# download_extra.ipynb
URLを貼るだけでモデルをDL・自動振り分けします。

**手順:** `download_list.txt` を編集 → Cell 1〜5 を順番に実行

## Cell 1: 設定・トークン読み込み

In [ ]:
# ===== Cell 1: 設定・トークン読み込み =====
import os, json, subprocess, shutil
from pathlib import Path
from datetime import datetime

WORK_DIR = "/workspace/runpod-slim"
MODEL_DIR = "/workspace/runpod-slim/ComfyUI/models"
LOG_FILE  = f"{WORK_DIR}/download_log.json"
LIST_FILE = f"{WORK_DIR}/download_list.txt"

# .env からトークン読み込み
ENV_FILE = f"{WORK_DIR}/.env"
def load_env():
    env = {}
    if os.path.exists(ENV_FILE):
        with open(ENV_FILE) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                env[k.strip()] = v.strip().strip('"').strip("'")
    return env

_env = load_env()
HF_TOKEN      = _env.get("HF_TOKEN", "")
CIVITAI_TOKEN = _env.get("CIVITAI_TOKEN", "")

try:
    import requests
except ImportError:
    subprocess.run(["pip", "install", "-q", "requests"], check=True, capture_output=True)
    import requests

# ログ読み込み
def load_log():
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE) as f:
            return json.load(f)
    return {}

def save_log(log):
    with open(LOG_FILE, "w") as f:
        json.dump(log, f, ensure_ascii=False, indent=2)

print("✅ Cell 1 完了")

## Cell 2: download_list.txt 読み込み

In [ ]:
# ===== Cell 2: download_list.txt 読み込み =====

# フォルダ自動判定（拡張子 + ファイル名から推定）
def guess_folder(filename):
    fn = filename.lower()
    if fn.endswith(".gguf"):                                                          return "diffusion_models"
    if "ae.safetensors" in fn:                                                        return "vae"
    if any(x in fn for x in ["vae"]):                                                 return "vae"
    if any(x in fn for x in ["clip", "t5", "text_encoder"]):                          return "text_encoders"
    if any(x in fn for x in ["upscale", "esrgan", "realesrgan", "ultrasharp", "4x-", "2x-", "8x-"]): return "upscale_models"
    if any(x in fn for x in ["controlnet"]):                                          return "controlnet"
    if any(x in fn for x in ["lora", "locon"]):                                       return "loras"
    if fn.endswith(".pt") or fn.endswith(".pth"):                                      return "ultralytics/bbox"
    if fn.endswith(".safetensors"):                                                    return "diffusion_models"
    return "checkpoints"
    return None

# CivitAI API でモデルタイプ・バージョン情報取得
def civitai_api(version_id):
    try:
        r = requests.get(
            f"https://civitai.com/api/v1/model-versions/{version_id}",
            timeout=10
        )
        if r.status_code != 200:
            return None
        return r.json()
    except Exception:
        return None

# CivitAI URL からバージョンID抽出
def parse_civitai_url(url):
    import re
    # api/download/models/XXXXX 形式
    m = re.search(r"api/download/models/(\d+)", url)
    if m:
        return m.group(1)
    # ?modelVersionId=XXXXX 形式
    m = re.search(r"modelVersionId=(\d+)", url)
    if m:
        return m.group(1)
    # /models/XXXXX/ の数字（モデルIDなのでバージョンIDではない）
    m = re.search(r"/models/(\d+)", url)
    if m:
        return m.group(1)
    return None

# CivitAI タイプ → フォルダマッピング
CIVITAI_TYPE_MAP = {
    "LORA":       "loras",
    "LoCon":      "loras",
    "Checkpoint": "checkpoints",
    "VAE":        "vae",
    "TextualInversion": "embeddings",
    "Upscaler":   "upscale_models",
    "ControlNet": "controlnet",
}

# リスト読み込み・パース
entries = []
errors  = []

if not os.path.exists(LIST_FILE):
    print(f"⚠️ {LIST_FILE} が見つかりません")
    print( "   download_list.txt を WORK_DIR に配置してください")
else:
    with open(LIST_FILE) as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = [p.strip() for p in line.split("\t")]
            if len(parts) < 3:
                errors.append(f"行{i}: カラム不足 → {line}")
                continue
            name    = parts[0]
            kind    = parts[1].lower()
            url     = parts[2]
            ver_id  = parts[3] if len(parts) > 3 and parts[3] not in ("-", "") else None
            dest_ov = parts[4] if len(parts) > 4 and parts[4] not in ("-", "") else None
            entries.append({"name": name, "kind": kind, "url": url,
                            "ver_id": ver_id, "dest_override": dest_ov, "line": i})

    if errors:
        print("⚠️ パースエラー:")
        for e in errors:
            print(f"  {e}")
    print(f"✅ Cell 2 完了 — {len(entries)} 件読み込み")

## Cell 3: DL実行

In [ ]:
# ===== Cell 3: DL実行（自動振り分け・スキップ・バージョン記録） =====

def _download(url, dest, label, token_header=None):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    tmp = dest + ".tmp"
    headers = token_header or {}
    try:
        with requests.get(url, headers=headers, stream=True, timeout=30) as r:
            r.raise_for_status()
            total = int(r.headers.get("Content-Length", 0))
            downloaded = 0
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        if total:
                            pct = downloaded / total * 100
                            print(f"  {downloaded/1e9:.2f}/{total/1e9:.2f} GB ({pct:.1f}%)", end="\r", flush=True)
        os.rename(tmp, dest)
        print(f"\n  ✅ {label}")
        return True
    except Exception as e:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise RuntimeError(f"{label}: {e}")

log = load_log()
dl_ok, dl_skip, dl_fail = [], [], []

for entry in entries:
    name    = entry["name"]
    kind    = entry["kind"]
    url     = entry["url"]
    dest_ov = entry["dest_override"]

    # --- Ollama は別処理 ---
    if kind == "ollama":
        model_name = url
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
        if model_name in result.stdout:
            dl_skip.append(name)
        else:
            print(f"⬇️  Ollama pull: {model_name}")
            r = subprocess.run(["ollama", "pull", model_name], capture_output=True, text=True)
            if r.returncode == 0:
                log[name] = {"kind": "ollama", "model": model_name,
                             "downloaded_at": datetime.now().isoformat()}
                dl_ok.append(name)
            else:
                print(f"❌ {name}: ollama pull 失敗")
                dl_fail.append(name)
        continue

    # --- HuggingFace ---
    if kind == "hf":
        filename = url.split("/")[-1].split("?")[0]
        folder   = dest_ov or guess_folder(filename) or "checkpoints"
        dest     = f"{MODEL_DIR}/{folder}/{filename}"
        if os.path.exists(dest):
            dl_skip.append(name)
            continue
        print(f"⬇️  HF: {filename} → {folder}/")
        headers = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}
        try:
            _download(url, dest, filename, headers)
            log[name] = {"kind": "hf", "url": url, "dest": dest,
                         "downloaded_at": datetime.now().isoformat()}
            dl_ok.append(name)
        except RuntimeError as e:
            print(f"❌ {e}")
            dl_fail.append(name)
        continue

    # --- CivitAI ---
    if kind == "civitai":
        ver_id = entry["ver_id"] or parse_civitai_url(url)
        if not ver_id:
            print(f"❌ {name}: バージョンID取得不可 → {url}")
            dl_fail.append(name)
            continue

        api_data = civitai_api(ver_id)
        if api_data:
            # ファイル名
            files = api_data.get("files", [])
            filename = files[0]["name"] if files else f"{name}.safetensors"
            # タイプ判定
            model_type = api_data.get("model", {}).get("type", "")
            folder = dest_ov or CIVITAI_TYPE_MAP.get(model_type) or guess_folder(filename) or "checkpoints"
            # 重複検知（同モデルの別バージョン）
            existing = list(Path(f"{MODEL_DIR}/{folder}").glob(f"{Path(filename).stem}*")) if Path(f"{MODEL_DIR}/{folder}").exists() else []
            if existing and not any(str(e).endswith(filename) for e in existing):
                print(f"⚠️  {name}: 別バージョンが存在 → {[e.name for e in existing]}")
        else:
            filename = f"{name}.safetensors"
            folder   = dest_ov or "checkpoints"
            print(f"⚠️  {name}: CivitAI API取得失敗 → {folder}/ に仮保存")

        dest = f"{MODEL_DIR}/{folder}/{filename}"
        if os.path.exists(dest):
            dl_skip.append(name)
            continue

        print(f"⬇️  CivitAI: {filename} → {folder}/")
        dl_url = f"https://civitai.com/api/download/models/{ver_id}"
        if CIVITAI_TOKEN:
            dl_url += f"?token={CIVITAI_TOKEN}"
        try:
            _download(dl_url, dest, filename)
            log[name] = {"kind": "civitai", "version_id": ver_id,
                         "dest": dest, "downloaded_at": datetime.now().isoformat()}
            dl_ok.append(name)
        except RuntimeError as e:
            print(f"❌ {e}")
            dl_fail.append(name)

save_log(log)

# 結果サマリー
print(f"\n✅ Cell 3 完了 — DL:{len(dl_ok)} / スキップ:{len(dl_skip)} / 失敗:{len(dl_fail)}")
if dl_fail:
    print("❌ 失敗リスト:")
    for n in dl_fail:
        print(f"  {n}")

## Cell 4: 更新チェック（CivitAI）

In [ ]:
# ===== Cell 4: 更新チェック（CivitAI 新バージョン確認） =====

log = load_log()
updates_found = []

for entry in entries:
    if entry["kind"] != "civitai":
        continue
    name   = entry["name"]
    ver_id = entry["ver_id"] or parse_civitai_url(entry["url"])
    if not ver_id:
        continue

    api_data = civitai_api(ver_id)
    if not api_data:
        continue

    # 最新バージョンID
    model_id = api_data.get("modelId")
    if not model_id:
        continue
    try:
        r = requests.get(f"https://civitai.com/api/v1/models/{model_id}", timeout=10)
        if r.status_code != 200:
            continue
        model_data = r.json()
        versions = model_data.get("modelVersions", [])
        if not versions:
            continue
        latest_id = str(versions[0].get("id", ""))
        if latest_id and latest_id != str(ver_id):
            updates_found.append({
                "name": name,
                "current": ver_id,
                "latest": latest_id,
                "model_url": f"https://civitai.com/models/{model_id}"
            })
    except Exception:
        continue

if updates_found:
    print("🆕 更新あり:")
    for u in updates_found:
        print(f"  {u['name']}: v{u['current']} → v{u['latest']}")
        print(f"    {u['model_url']}")
    print("\n  download_list.txt のバージョンIDを更新して Cell 3 を再実行してください")
else:
    print("✅ Cell 4 完了 — 全て最新")

## Cell 5: ワークフロー自動置換

In [ ]:
# ===== Cell 5: ワークフロー自動置換 =====

import glob as _glob

WORKFLOWS_DIR = f"{WORK_DIR}/workflows"
log = load_log()

# name → 新しいファイル名のマップを構築
name_to_filename = {}
for entry in entries:
    name = entry["name"]
    if name not in log:
        continue
    dest = log[name].get("dest", "")
    if dest:
        name_to_filename[name] = Path(dest).name

if not name_to_filename:
    print("⚠️ 置換対象なし（Cell 3 を先に実行してください）")
else:
    wf_files = _glob.glob(f"{WORKFLOWS_DIR}/*.json")
    if not wf_files:
        print(f"⚠️ ワークフローが見つかりません: {WORKFLOWS_DIR}/")
    else:
        replaced_any = False
        errors = []
        for wf_path in wf_files:
            try:
                with open(wf_path, encoding="utf-8") as f:
                    content = f.read()
                original = content
                for name, new_fname in name_to_filename.items():
                    # ログの旧ファイル名と一致する箇所を置換
                    old_entry = log.get(name, {})
                    old_dest  = old_entry.get("prev_dest", old_entry.get("dest", ""))
                    if old_dest:
                        old_fname = Path(old_dest).name
                        if old_fname in content and old_fname != new_fname:
                            content = content.replace(old_fname, new_fname)
                            replaced_any = True
                if content != original:
                    with open(wf_path, "w", encoding="utf-8") as f:
                        f.write(content)
            except Exception as e:
                errors.append(f"{Path(wf_path).name}: {e}")

        if errors:
            print("❌ 置換エラー:")
            for e in errors:
                print(f"  {e}")
        elif replaced_any:
            print("✅ Cell 5 完了 — ワークフロー置換完了")
        else:
            print("✅ Cell 5 完了 — 置換対象なし（既に最新）")